# How to calculate the Grundrentenzuschlag

The Grundrentenzuschlag (basic pension supplement) raises the statutory pensions of
people who paid contributions for at least 33 years at low earnings. It is reduced by
the pensioner's income (§ 97a SGB VI). This income is the sum of

- the zu versteuerndes Einkommen (taxable income),
- the steuerfreier Teil (tax-free part) of pensions, and
- capital income taxed under the Abgeltungsteuer,

taken as one twelfth per month. Income up to a lower threshold is not credited; 60 % of
the income between the lower and the upper threshold is credited; income above the
upper threshold is credited in full. The thresholds are multiples of the Rentenwert
(36.56 and 46.78 times the Rentenwert for singles, 57.03 and 67.27 times for married
people), rounded up to a full Euro. A calendar year uses the Rentenwert of 1 July of
the previous year.

The Grundrentenzuschlag does not use current income but the income of an earlier year,
which GETTSIM calculates in a separate run. This document shows how.

### Overview

The calculation involves two steps:

1. Calculate the income relevant for the Grundrentenzuschlag in the **previous** year.
   Use the target  
   `("sozialversicherung", "rente", "grundrente", "einkommen_für_einkommensberechnung_im_folgejahr_m")`  
   with the policy environment of the previous year.

2. Use the result as the input  
   `("sozialversicherung", "rente", "grundrente", "einkommen_vorjahr_m")`  
   and calculate the Grundrentenzuschlag `("sozialversicherung", "rente", "grundrente", "betrag_m")`
   with the policy environment of the current year.

> **Note**: GETTSIM simplifies the statute in a few ways:
>
> - The Deutsche Rentenversicherung uses the income of the year two years before (in
>   the year the pension starts) or three years before (from the first annual check
>   on). GETTSIM's target is meant to be run for the previous year. If you need the
>   statutory lag, run step 1 for the right year yourself.
> - If the tax office has no data on a person, the statute cuts gross pensions by fixed
>   percentages instead. GETTSIM does not model this fallback.
> - The zu versteuerndes Einkommen and the capital income of jointly assessed spouses
>   are split equally between them. Only the couple's sum enters the crediting, so the
>   split does not change the result.
> - The tax-free part of Versorgungsbezüge (civil servant pensions) is not modelled.

## Example

Consider a single pensioner, born in 1953, who has drawn her statutory pension since
2018. In 2025, she received a statutory pension of €1,250 and an occupational pension
(Betriebsrente) of €500 per month, plus €3,000 of capital income. She received no
Grundrentenzuschlag in 2025. We want her Grundrentenzuschlag in January 2026.

### Step 1: Income in the previous year

In [ ]:
import numpy as np

from gettsim import InputData, MainTarget, TTTargets, main

First, we compile the data needed to calculate her income tax for 2025.

In [ ]:
input_data_2025 = {
    "alter": np.array([72]),
    "alter_monate": np.array([864]),
    "arbeitsstunden_w": np.array([0]),
    "behinderungsgrad": np.array([0]),
    "einkommensteuer": {
        "abzüge": {
            "beitrag_private_rentenversicherung_m": np.array([0]),
            "kinderbetreuungskosten_m": np.array([0]),
            "p_id_kinderbetreuungskostenträger": np.array([-1]),
        },
        "einkünfte": {
            "aus_forst_und_landwirtschaft": {"betrag_y": np.array([0])},
            "aus_gewerbebetrieb": {"betrag_y": np.array([0])},
            "aus_nichtselbstständiger_arbeit": {
                "tatsächliche_werbungskosten_y": np.array([0]),
            },
            "aus_selbstständiger_arbeit": {"betrag_y": np.array([0])},
            "aus_vermietung_und_verpachtung": {"betrag_y": np.array([0])},
            "ist_hauptberuflich_selbstständig": np.array([False]),
            "sonstige": {
                "alle_weiteren_y": np.array([0]),
                "rente": {
                    "alter_beginn_leistungsbezug_sonstige_private_vorsorge": np.array(
                        [65]
                    ),
                },
            },
        },
        "gemeinsam_veranlagt": np.array([False]),
    },
    "einnahmen": {
        "bruttolohn_m": np.array([0]),
        "kapitalerträge_y": np.array([3000]),
        "renten": {
            "aus_berufsständischen_versicherungen_m": np.array([0]),
            "basisrente_m": np.array([0]),
            "betriebliche_altersvorsorge_m": np.array([500]),
            "geförderte_private_vorsorge_m": np.array([0]),
            "gesetzliche_m": np.array([1250]),
            "sonstige_private_vorsorge_m": np.array([0]),
        },
    },
    "familie": {
        "alleinerziehend": np.array([False]),
        "p_id_ehepartner": np.array([-1]),
        "p_id_elternteil_1": np.array([-1]),
        "p_id_elternteil_2": np.array([-1]),
    },
    "geburtsjahr": np.array([1953]),
    "kindergeld": {
        "in_ausbildung": np.array([False]),
        "p_id_empfänger": np.array([-1]),
    },
    "p_id": np.array([0]),
    "sozialversicherung": {
        "kranken": {"beitrag": {"privat_versichert": np.array([False])}},
        "pflege": {"beitrag": {"hat_kinder": np.array([True])}},
        "rente": {
            "altersrente": {"betrag_m": np.array([1250.0])},
            "bezieht_altersrente": np.array([True]),
            "grundrente": {"betrag_m": np.array([0.0])},
            "jahr_renteneintritt": np.array([2018]),
            "verzichtet_auf_versicherungsfreiheit_wegen_alters": np.array([False]),
            "verzichtet_auf_versicherungsfreiheit_wegen_geringfügiger_beschäftigung": np.array(
                [False]
            ),
        },
    },
}

We pass the Grundrentenzuschlag of 2025 as data because it is tax-free (§ 3 Nr. 14a
EStG): GETTSIM subtracts it from the taxable pension. Now, we calculate the income
relevant for the Grundrentenzuschlag in 2026, using the policy environment of 2025.

In [ ]:
einkommen_2025 = main(
    main_target=MainTarget.results.tree,
    policy_date_str="2025-01-01",
    tt_targets=TTTargets.tree(
        {
            "sozialversicherung": {
                "rente": {
                    "grundrente": {
                        "einkommen_für_einkommensberechnung_im_folgejahr_m": None,
                    }
                }
            }
        }
    ),
    input_data=InputData.tree(input_data_2025),
    include_warn_nodes=False,
)["sozialversicherung"]["rente"]["grundrente"][
    "einkommen_für_einkommensberechnung_im_folgejahr_m"
]
einkommen_2025

This is less than her gross income of €1,750 per month plus capital income, because
the zu versteuerndes Einkommen deducts, e.g., health and long-term care insurance
contributions and the Sparer-Pauschbetrag.

### Step 2: Grundrentenzuschlag in the current year

Now we calculate the Grundrentenzuschlag for January 2026 with the policy environment
of that date. Her pension record has 35 years (420 months) of Grundrentenzeiten, which
are also her Grundrentenbewertungszeiten, with 17.5 Entgeltpunkte (0.5 per year) in
total. We pass a Zugangsfaktor of 1 as data, so we need not supply her full insurance
history.

In [ ]:
input_data_2026 = {
    "familie": {"p_id_ehepartner": np.array([-1])},
    "p_id": np.array([0]),
    "sozialversicherung": {
        "rente": {
            "altersrente": {"zugangsfaktor": np.array([1.0])},
            "grundrente": {
                "bewertungszeiten_monate": np.array([420]),
                "einkommen_vorjahr_m": einkommen_2025,
                "grundrentenzeiten_monate": np.array([420]),
                "mean_entgeltpunkte": np.array([17.5]),
            },
        },
    },
}

main(
    main_target=MainTarget.results.tree,
    policy_date_str="2026-01-01",
    tt_targets=TTTargets.tree(
        {
            "sozialversicherung": {
                "rente": {
                    "grundrente": {
                        "basisbetrag_m": None,
                        "anzurechnendes_einkommen_m": None,
                        "betrag_m": None,
                    }
                }
            }
        }
    ),
    input_data=InputData.tree(input_data_2026),
    include_warn_nodes=False,
)

Her income lies between the 2026 thresholds for singles: 36.56 × €40.79 = €1,491.28,
rounded up to €1,492, and 46.78 × €40.79 = €1,908.16, rounded up to €1,909. So 60 % of
the income above €1,492 is credited: 0.6 × (€1,646.50 − €1,492) = €92.70. This amount is
subtracted from the Grundrentenzuschlag before crediting (`basisbetrag_m`), which leaves
€375.19 − €92.70 = €282.49.